# Notebook 01: Data Acquisition and Preprocessing

This notebook prepares the labelled benign and phishing domain datasets for downstream feature engineering, modelling, and evaluation. It loads the source files, normalises domain records, aligns shared WHOIS/RDAP, DNS, TLS, and enrichment fields, handles source-format differences, removes duplicate records, and exports the cleaned combined dataset used by the later notebooks.

## 1. Configuration

Project paths, dataset registry entries, and shared parsing utilities.

In [1]:
from __future__ import annotations
import math
import os
import re
from pathlib import Path
from typing import Any, Callable, Iterable, Optional, Sequence
import ijson
import pandas as pd
import tldextract

# Regex patterns used to clean domain strings
_DOMAIN_PROTOCOL_RE = re.compile(r"^[a-zA-Z][a-zA-Z0-9+\-.]*://")
_DOMAIN_WHITESPACE_RE = re.compile(r"\s+")

# Public suffix extractor used to identify registered domains
_TLD_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())

# Possible TLS fields for extraction indicating a successful scan
_TLS_SUCCESS_KEYS = (
    "handshake_success",
    "tls_handshake_success",
    "success",
    "completed",
    "reachable",
)

# Possible TLS fields for extraction indicating scan errors or failures
_TLS_ERROR_KEYS = (
    "error",
    "errors",
    "scan_error",
    "tls_error",
    "handshake_error",
    "failure_reason",
)

# Converts MongoDB extended JSON values into standard Python/Pandas values
def unwrap_mongo_extended_json(value: Any) -> Any:
    if not isinstance(value, dict):
        return value
    if "$date" in value:
        date_value = value["$date"]
        if isinstance(date_value, dict) and "$numberLong" in date_value:
            try:
                return pd.to_datetime(int(date_value["$numberLong"]), unit="ms", errors="coerce", utc=True)
            except Exception:
                return pd.NaT
        return date_value
    if "$numberLong" in value:
        try:
            return int(value["$numberLong"])
        except Exception:
            return value["$numberLong"]
    if "$numberInt" in value:
        try:
            return int(value["$numberInt"])
        except Exception:
            return value["$numberInt"]
    if "$numberDouble" in value:
        try:
            return float(value["$numberDouble"])
        except Exception:
            return value["$numberDouble"]
    return value

# Safely parses dates into timezone-free UTC Pandas timestamps
def parse_datetime_safe(value: Any) -> pd.Timestamp:
    value = unwrap_mongo_extended_json(value)
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return pd.NaT
    try:
        timestamp = pd.to_datetime(value, errors="coerce", utc=True)
    except Exception:
        return pd.NaT
    if pd.isna(timestamp):
        return pd.NaT
    return timestamp.tz_convert("UTC").tz_localize(None)

# Cleans and standardises domain strings
def normalise_domain(value: Any) -> Optional[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip().lower()
    if not text:
        return None
    text = _DOMAIN_PROTOCOL_RE.sub("", text)
    text = text.split("/", 1)[0].split("?", 1)[0].split("#", 1)[0]
    if "@" in text:
        text = text.rsplit("@", 1)[-1]
    text = text.split(":", 1)[0]
    text = text.lstrip(".").rstrip(".")
    if text.startswith("www."):
        text = text[4:]
    text = _DOMAIN_WHITESPACE_RE.sub("", text)
    return text or None

# Extracts the registered domain using the Public Suffix List
def get_registered_domain(domain: Any) -> str:
    normalised = normalise_domain(domain) or ""
    if not normalised:
        return ""
    extracted = _TLD_EXTRACTOR(normalised)
    if not extracted.suffix or not extracted.domain:
        return normalised
    return f"{extracted.domain}.{extracted.suffix}"

# Raises an error if required DataFrame columns are missing
def assert_required_columns(
    df: pd.DataFrame,
    required_columns: Sequence[str],
    label: str,
) -> None:
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f"[{label}] Missing required columns: {missing}")

# Checks whether a TLS handshake succeeded from available TLS metadata
def infer_tls_handshake_success(tls_value: Any) -> Any:
    if not isinstance(tls_value, dict):
        return pd.NA
    for key in _TLS_SUCCESS_KEYS:
        if key in tls_value and tls_value.get(key) is not None:
            return int(bool(tls_value.get(key)))
    for key in _TLS_ERROR_KEYS:
        error_value = tls_value.get(key)
        if isinstance(error_value, (list, tuple, set, dict)) and len(error_value) > 0:
            return 0
        if isinstance(error_value, str) and error_value.strip():
            return 0
        if error_value:
            return 0
    return pd.NA

# Checks whether a TLS scan failed using TLS errors and handshake status
def infer_tls_scan_failed(tls_value: Any, handshake_success: Any) -> Any:
    if not isinstance(tls_value, dict):
        return 1
    if pd.isna(handshake_success):
        for key in _TLS_ERROR_KEYS:
            error_value = tls_value.get(key)
            if isinstance(error_value, str) and error_value.strip():
                return 1
            if isinstance(error_value, (list, tuple, set, dict)) and len(error_value) > 0:
                return 1
            if error_value:
                return 1
        return 0
    return int(not bool(handshake_success))

# Adds a registered_domain column derived from the domain column
def add_registered_domain_column(
    df: pd.DataFrame,
    domain_col: str = "domain",
    output_col: str = "registered_domain",
    registered_domain_getter: Callable[[Any], str] = get_registered_domain,
) -> pd.DataFrame:
    enriched = df.copy()
    enriched[output_col] = enriched[domain_col].map(registered_domain_getter)
    return enriched

# Builds a per-source missingness summary for selected features
def build_source_missingness_table(
    df: pd.DataFrame,
    feature_columns: Sequence[str],
    source_col: str = "source_dataset",
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for source_value, source_df in df.groupby(source_col, dropna=False):
        total_rows = len(source_df)
        for column in feature_columns:
            series = source_df[column] if column in source_df.columns else pd.Series(pd.NA, index=source_df.index)
            missing_mask = series.isna()
            if not pd.api.types.is_numeric_dtype(series):
                missing_mask = missing_mask | series.astype(str).str.strip().eq("")
            missing_count = int(missing_mask.sum())
            rows.append(
                {
                    source_col: source_value,
                    "feature": column,
                    "missing_count": missing_count,
                    "missing_ratio": float(missing_count / total_rows) if total_rows else 0.0,
                }
            )
    return pd.DataFrame(rows).sort_values(
        [source_col, "missing_ratio", "feature"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

# Finds the project data root using an environment variable
def resolve_data_root(
    env_var: str = "PHISHING_PROJECT_DATA_ROOT",
    preferred: Optional[Iterable[Path]] = None,
) -> Path:
    env_value_raw = os.environ.get(env_var, "").strip()
    env_path = Path(env_value_raw) if env_value_raw else None
    candidates: list[Path] = []
    if env_path is not None:
        candidates.append(env_path)
    if preferred is not None:
        candidates.extend(preferred)
    for candidate in candidates:
        if candidate.exists():
            return candidate

# Notebook display options
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

In [2]:
# Define the data directory
DATA_ROOT     = resolve_data_root(preferred=[Path("/datasets"), Path("datasets"), Path(".")])

# Define the raw input and processed output folders
RAW_DIR       = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"

# Create the processed folder if it does not already exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Define input datasets with their paths, source names, and labels
DATASET_REGISTRY: list[dict[str, Any]] = [
    {
        "name": "benign_umbrella",
        "path": RAW_DIR / "benign_umbrella.json",
        "source_dataset": "benign_umbrella",
        "label": 0,
    },
    {
        "name": "phishing",
        "path": RAW_DIR / "phishing.json",
        "source_dataset": "phishing",
        "label": 1,
    },
]

In [3]:
# Fields to keep from raw source records
FIELDS_OF_INTEREST: list[str] = [
    # Domain identifier fields
    "domain_name", "domain", "name", "host", "url", "fqdn",

    # Observation or scan timestamp fields
    "evaluated_on", "sourced_on", "observed_at", "timestamp", "date",
    "scan_date", "captured_at", "created",

    # RDAP registration dates
    "rdap.registration_date", "rdap.expiration_date",

    # WHOIS and generic registration/expiry dates
    "registration_date", "expiration_date",
    "whois.creation_date", "whois.expiration_date",
    "creation_date", "created_date",
    "expires", "expire_date",

    # Domain status and nameserver fields
    "rdap.nameservers", "rdap.status", "status", "domain_status",

    # DNS, TLS, and TLS evaluation metadata
    "dns", "tls", "remarks.tls_evaluated_on",
]

# Reads nested dictionary values using dotted paths e.g. rdap.status
def _deep_get(obj: dict, dotted_key: str) -> Any:                                                          
    parts = dotted_key.split(".")
    cur = obj
    for p in parts:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return None
    return cur
# Extracts only the selected fields from one raw record
def _extract_row(record: dict, fields: list[str]) -> dict:                                                  
    row: dict[str, Any] = {}
    for f in fields:
        val = _deep_get(record, f)
        if val is not None:
            row[f] = val
    return row
    
# Finds where records are located inside a JSON file
def _json_record_prefix(path: Path) -> str:
    with open(path, "rb") as fh:
        parser = ijson.parse(fh)
        try:
            prefix, event, value = next(parser)
        except StopIteration:
            return ""
        # Handles files that are a top-level JSON array
        if event == "start_array":
            return "item" 
        # Rejects unsupported top-level structures
        if event != "start_map":
            return ""
        # Handles files with records stored under common keys
        for prefix, event, value in parser:
            if prefix == "" and event == "map_key" and value in ("records", "data", "results", "items"):
                key = value
                _, next_event, _ = next(parser)
                if next_event == "start_array":
                    return f"{key}.item"
                return ""
            if prefix == "" and event == "end_map":
                return ""
    return ""

# Streams JSON records and keeps only required fields
def _load_json_streaming(path: Path) -> list[dict]:                                                     
    record_prefix = _json_record_prefix(path)
    slim: list[dict] = []
    with open(path, "rb") as fh:
        records = ijson.items(fh, record_prefix)
        for record in records:
            slim.append(_extract_row(record, FIELDS_OF_INTEREST))
    return slim

# Loads CSV, JSON, or Excel source files into a DataFrame
def load_dataset(path: Path) -> pd.DataFrame:                                               
    path = Path(path)
    suffix = path.suffix.lower()
    # Stream JSON files to reduce memory usage
    if suffix == ".json":
        slim_rows = _load_json_streaming(path)
        df = pd.DataFrame(slim_rows)
        del slim_rows
        return df

In [4]:
# Output columns used to store up to four nameservers
RAW_NAMESERVER_COLUMNS = ["name_server_1", "name_server_2", "name_server_3", "name_server_4"]

# Output columns used to store up to four domain status values
RAW_STATUS_COLUMNS = ["domain_status_1", "domain_status_2", "domain_status_3", "domain_status_4"]

# Cleans nameserver values by lowercasing and removing empty entries
def normalise_nameserver_values(values: list[Any]) -> list[str]:
    cleaned: list[str] = []
    for value in values:
        text = str(value).strip().lower()
        if text:
            cleaned.append(text)
    return cleaned

# Converts status values into a clean lowercase list
def normalise_status_values(values: Any) -> list[str]:
    if isinstance(values, list):
        candidates = values
    # Split string statuses on commas, semicolons, or whitespace
    elif isinstance(values, str):
        candidates = [part.strip() for part in re.split(r"[,;\s]+", values) if part.strip()]
    else:
        candidates = []
    cleaned: list[str] = []
    for value in candidates:
        text = str(value).strip().lower()
        if text:
            cleaned.append(text)
    return cleaned

# Creates nameserver count and first four nameserver feature columns
def add_nameserver_features(df: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame(index=df.index)
    # Use RDAP nameservers when available. If not available, empty lists are used
    if "rdap.nameservers" in df.columns:
        ns_values = df["rdap.nameservers"].apply(lambda value: normalise_nameserver_values(value if isinstance(value, list) else []))
    else:
        ns_values = pd.Series([[] for _ in range(len(df))], index=df.index)
    # Count nameservers and expose up to four raw values
    features["nameserver_count"] = ns_values.map(len).astype(int)
    for idx, column in enumerate(RAW_NAMESERVER_COLUMNS):
        features[column] = ns_values.map(lambda values, idx=idx: values[idx] if idx < len(values) else "")
    return features

# Creates status count and first four domain status feature columns
def add_status_features(df: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame(index=df.index)
    # Prefer RDAP status, then generic status fields
    status_source = next((column for column in ["rdap.status", "status", "domain_status"] if column in df.columns), None)
    if status_source is not None:
        status_values = df[status_source].apply(normalise_status_values)
    else:
        status_values = pd.Series([[] for _ in range(len(df))], index=df.index)
    # Count statuses and expose up to four raw values
    features["status_count"] = status_values.map(len).astype(int)
    for idx, column in enumerate(RAW_STATUS_COLUMNS):
        features[column] = status_values.map(lambda values, idx=idx: values[idx] if idx < len(values) else "")
    return features

In [5]:
# Default DNS feature values used when DNS data is missing
DNS = {
    "has_a_record": 0,
    "a_record_count": 0,
    "aaaa_record_count": 0,
    "mx_record_count": 0,
    "ns_record_count": 0,
    "txt_record_count": 0,
    "has_soa_record": 0,
    "soa_present": 0,
    "dns_lookup_failed": 1,
}

# Default TLS feature values used when TLS data is missing
TLS = {
    "tls_handshake_success": pd.NA,
    "cert_present": 0,
    "cert_expired": pd.NA,
    "cert_days_until_expiry": pd.NA,
    "cert_validity_days": pd.NA,
    "cert_issuer_org": pd.NA,
    "cert_subject_cn_matches_domain": pd.NA,
    "cert_san_count": pd.NA,
    "tls_scan_failed": pd.NA,
    "tls_no_certificate": pd.NA,
}

# Final column order for the cleaned modelling dataset
FINAL_SCHEMA_COLUMNS = [
    "source_dataset",
    "label",
    "domain",
    "registered_domain",
    "observed_at",
    "registration_date",
    "expiration_date",
    *RAW_NAMESERVER_COLUMNS,
    "nameserver_count",
    *RAW_STATUS_COLUMNS,
    "status_count",
    *DNS,
    *TLS,
]

# Candidate raw columns for each standard output field
FIELD_MAP: dict[str, list[str]] = {
    "domain": ["domain_name", "domain", "name", "host", "url", "fqdn"],
    "observed_at": ["evaluated_on", "sourced_on", "observed_at", "timestamp", "date", "scan_date", "captured_at", "created"],
    "registration_date": ["rdap.registration_date", "registration_date", "whois.creation_date", "creation_date", "created_date"],
    "expiration_date": ["rdap.expiration_date", "expiration_date", "whois.expiration_date", "expires", "expire_date"],
}

# Returns the first available column from a list of candidates
def _resolve_field(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    return next((col for col in candidates if col in df.columns), None)

# Counts items in list-like values
def _as_count(value: Any) -> int:
    if isinstance(value, dict):
        return len(value)
    if isinstance(value, (list, tuple, set)):
        return len(value)
    return 0 if value is None else 1

# Returns 1 when a value contains usable data
def _is_present(value: Any) -> int:
    return int(isinstance(value, dict) or _as_count(value) > 0)

# Extracts DNS-related features from one row
def _dns_feature_row(row: pd.Series) -> dict[str, Any]:
    features = DNS.copy()
    dns = row.get("dns")
    # Return default values if DNS data is unavailable
    if not isinstance(dns, dict):
        return features
    a_count = _as_count(dns.get("A"))
    soa = dns.get("SOA")
    zone_soa = dns.get("zone_SOA")
    # Populate DNS record counts and lookup status
    features.update({
        "has_a_record": int(a_count > 0),
        "a_record_count": a_count,
        "aaaa_record_count": _as_count(dns.get("AAAA")),
        "mx_record_count": _as_count(dns.get("MX")),
        "ns_record_count": _as_count(dns.get("NS")),
        "txt_record_count": _as_count(dns.get("TXT")),
        "has_soa_record": _is_present(soa),
        "soa_present": int(_is_present(soa) or _is_present(zone_soa)),
        "dns_lookup_failed": 0,
    })
    return features

# Selects the leaf certificate from a certificate chain
def _leaf_certificate(certs: Any) -> Optional[dict]:
    if not isinstance(certs, list) or not certs:
        return None
    leaf = next((cert for cert in reversed(certs) if isinstance(cert, dict) and not cert.get("is_root", False)), None)
    return leaf or (certs[-1] if isinstance(certs[-1], dict) else None)

# Calculates certificate validity length in days
def _certificate_validity_days(cert: Optional[dict]) -> Any:
    if not cert:
        return pd.NA
    # Use provided validity length when available
    if cert.get("valid_len") is not None:
        valid_len = pd.to_numeric(cert.get("valid_len"), errors="coerce")
        if pd.isna(valid_len):
            return pd.NA
        # Convert seconds to days when needed
        return float(valid_len) / 86_400.0 if float(valid_len) > 10_000 else float(valid_len)
    # Otherwise calculate validity from start and end dates    
    start = parse_datetime_safe(cert.get("validity_start"))
    end = parse_datetime_safe(cert.get("validity_end"))
    if pd.isna(start) or pd.isna(end):
        return pd.NA
    return (end - start).days

# Counts DNS subject alternative names in the certificate
def _certificate_san_count(cert: Optional[dict]) -> Any:
    if not cert:
        return pd.NA
    subject_alt_name = next((ext.get("value") or "" for ext in cert.get("extensions", []) if isinstance(ext, dict) and ext.get("name") == "subjectAltName"), "")
    return subject_alt_name.count("DNS:")

# Extracts TLS and certificate features from one row
def _tls_feature_row(row: pd.Series, domain: Optional[str]) -> dict[str, Any]:
    features = TLS.copy()
    tls = row.get("tls")
    # Return default values if TLS data is unavailable
    if not isinstance(tls, dict):
        return features
    cert = _leaf_certificate(tls.get("certificates"))
    cert_present = int(cert is not None)
    handshake_success = infer_tls_handshake_success(tls)
    tls_scan_failed = infer_tls_scan_failed(tls, handshake_success)
    # Choose a reference date for certificate expiry checks
    reference_date = parse_datetime_safe(row.get("remarks.tls_evaluated_on"))
    if pd.isna(reference_date):
        reference_date = parse_datetime_safe(row.get("evaluated_on"))
    expiry_date = parse_datetime_safe(cert.get("validity_end")) if cert else pd.NaT
    has_expiry_ref = not pd.isna(expiry_date) and not pd.isna(reference_date)
    # Check whether the certificate common name matches the domain
    common_name = normalise_domain(cert.get("common_name")) if cert else None
    domain_match = pd.NA
    if common_name and domain:
        domain_match = int(domain == common_name or domain.endswith(f".{common_name}"))
    # Flag successful TLS scans that returned no certificate
    tls_no_certificate = pd.NA
    if not pd.isna(handshake_success) and int(handshake_success) == 1:
        tls_no_certificate = int(not cert_present)

    # Populate TLS and certificate feature values
    features.update({
        "tls_handshake_success": handshake_success,
        "cert_present": cert_present,
        "cert_expired": pd.NA if not has_expiry_ref else int(expiry_date < reference_date),
        "cert_days_until_expiry": pd.NA if not has_expiry_ref else (expiry_date - reference_date).days,
        "cert_validity_days": _certificate_validity_days(cert),
        "cert_issuer_org": cert.get("organization") if cert and cert.get("organization") else pd.NA,
        "cert_subject_cn_matches_domain": domain_match,
        "cert_san_count": _certificate_san_count(cert),
        "tls_scan_failed": tls_scan_failed,
        "tls_no_certificate": tls_no_certificate,
    })
    return features

# Aligns a raw dataset to the final cleaned schema
def align_to_final_schema(df: pd.DataFrame, source_dataset: str, label: int) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    # Resolve and normalise the main domain field
    domain_col = _resolve_field(df, FIELD_MAP["domain"]) or next(iter(df.columns), None)
    out["source_dataset"] = source_dataset
    out["label"] = label
    out["domain"] = df[domain_col].apply(normalise_domain)
    out["registered_domain"] = out["domain"].map(get_registered_domain)
    # Parse standard date fields from available raw columns
    for col in ("observed_at", "registration_date", "expiration_date"):
        raw_col = _resolve_field(df, FIELD_MAP[col])
        if raw_col is None:
            print(f"  {col:30s} <- NOT FOUND")
            out[col] = pd.NaT
            continue
        print(f"  {col:30s} <- RAW COL '{raw_col}'")
        out[col] = df[raw_col].apply(parse_datetime_safe)
        print(f"  {'':<30s} parsed OK: {out[col].notna().sum():,} / {len(df):,}")
    # Add nameserver and status features
    nameserver_features = add_nameserver_features(df)
    for col in [*RAW_NAMESERVER_COLUMNS, "nameserver_count"]:
        out[col] = nameserver_features[col]
    status_features = add_status_features(df)
    for col in [*RAW_STATUS_COLUMNS, "status_count"]:
        out[col] = status_features[col]
    # Add DNS and TLS feature rows
    feature_rows = pd.concat([
        df.apply(_dns_feature_row, axis=1, result_type="expand"),
        df.apply(lambda row: _tls_feature_row(row, out.at[row.name, "domain"]), axis=1, result_type="expand"),
    ], axis=1)
    # Combine all fields and remove rows without a valid domain
    final_df = pd.concat([out, feature_rows], axis=1)[FINAL_SCHEMA_COLUMNS].copy()
    final_df = final_df[final_df["domain"].notna() & final_df["domain"].astype(str).ne("")].reset_index(drop=True)
    return final_df

In [6]:
# Checks that a DataFrame matches the final expected schema
def validate_schema(df: pd.DataFrame, label: str = "dataset") -> None:
    assert_required_columns(df, FINAL_SCHEMA_COLUMNS, label)
    print(f"[{label}] Schema OK")

## 2. Data Loading

In [7]:
# Load raw benign dataset using the first registry entry
benign_entry = DATASET_REGISTRY[0]
df_benign_raw = load_dataset(benign_entry["path"])

In [8]:
# Load raw phishing dataset using the second registry entry
phishing_entry = DATASET_REGISTRY[1]
df_phishing_raw = load_dataset(phishing_entry["path"])

## 3. Preprocessing

Domains are normalised, dates are parsed, and source-specific fields are aligned to a common modelling schema.


In [9]:
# Align benign data to the final schema and validate it
df_benign = align_to_final_schema(df_benign_raw, benign_entry["source_dataset"], benign_entry["label"])
validate_schema(df_benign, "benign")

  observed_at                    <- RAW COL 'evaluated_on'
                                 parsed OK: 368,956 / 368,956
  registration_date              <- RAW COL 'rdap.registration_date'
                                 parsed OK: 189,661 / 368,956
  expiration_date                <- RAW COL 'rdap.expiration_date'
                                 parsed OK: 187,857 / 368,956
[benign] Schema OK


In [10]:
# Align phishing data to the final schema and validate it
df_phishing = align_to_final_schema(df_phishing_raw, phishing_entry["source_dataset"], phishing_entry["label"])
validate_schema(df_phishing, "phishing")

  observed_at                    <- RAW COL 'evaluated_on'
                                 parsed OK: 164,425 / 164,425
  registration_date              <- RAW COL 'rdap.registration_date'
                                 parsed OK: 147,932 / 164,425
  expiration_date                <- RAW COL 'rdap.expiration_date'
                                 parsed OK: 145,056 / 164,425
[phishing] Schema OK


In [11]:
# Combine benign and phishing rows into one modelling dataset
df_combined = pd.concat([df_benign, df_phishing], ignore_index=True)

## 4. Outputs

Data-quality checks for missingness, duplicate control, and class balance.

In [12]:
# Store row count before removing duplicate records
before_dedup = len(df_combined)

# Remove duplicate rows based on domain, label, and source dataset
df_combined = df_combined.drop_duplicates(
    subset=["domain", "label", "source_dataset"],
).reset_index(drop=True)

# Store row count after deduplication
after_dedup = len(df_combined)

# Columns used to audit missingness by source dataset
MISSINGNESS_AUDIT_COLUMNS = [
    "domain",
    "registered_domain",
    "observed_at",
    "registration_date",
    "expiration_date",
    "nameserver_count",
    "status_count",
    "has_a_record",
    "dns_lookup_failed",
    "tls_handshake_success",
    "tls_scan_failed",
    "cert_present",
]

# Build and save a missingness audit report
source_missingness_df = build_source_missingness_table(
    df_combined,
    MISSINGNESS_AUDIT_COLUMNS,
    source_col="source_dataset",
)
source_missingness_df.to_csv(PROCESSED_DIR / "source_missingness_audit.csv", index=False)

## 5. Export

In [13]:
# Split the combined dataset back into benign and phishing subsets
df_benign_clean   = df_combined[df_combined["label"] == 0].copy()
df_phishing_clean = df_combined[df_combined["label"] == 1].copy()

# Save cleaned datasets and track output file paths
output_files: dict[str, Path] = {}
for tag, frame in [
    ("benign_cleaned",   df_benign_clean),
    ("phishing_cleaned", df_phishing_clean),
    ("combined_cleaned", df_combined),
]:
    csv_path = PROCESSED_DIR / f"{tag}.csv"
    frame.to_csv(csv_path, index=False)
    output_files[f"{tag}.csv"] = csv_path

# Add missingness audit path to output file registry
output_files["source_missingness_audit.csv"] = PROCESSED_DIR / "source_missingness_audit.csv"

In [14]:
# Calculate basic processing summary metrics
rows_loaded = len(df_benign_raw) + len(df_phishing_raw)
duplicates_removed = before_dedup - after_dedup

# Create summary table
pd.DataFrame({
    "metric": [
        "Rows loaded",
        "Rows retained",
        "Duplicates removed",
    ],
    "value": [
        rows_loaded,
        len(df_combined),
        duplicates_removed,
    ],
})

,metric,value
0,Rows loaded,533381
1,Rows retained,505539
2,Duplicates removed,27842
